# STEP 3.1 — 저장된 CVD / ALD 헤드 추론
STEP 3과 현재 evaluate가 저장한 헤드를 사용합니다. 이 노트북에서는 학습하지 않습니다.


In [ ]:
# 처음 한 번 PROJECT_PATH에 프로젝트 루트를 지정합니다. None은 저장된 설정 / 자동 탐색.
PROJECT_PATH = None  # 예: r"D:\GEVIS"
from pathlib import Path
import runpy
_source_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'notebooks/training/workflow.py').is_file())
workflow = runpy.run_path(str(_source_root / 'notebooks/training/workflow.py'))
ROOT = PROJECT_ROOT = workflow['configure'](PROJECT_PATH)


In [ ]:
DOMAIN = 'PECVD'  # PECVD 또는 PEALD
CHECKPOINTS = None


In [ ]:
import pandas as pd
from pecvd.gevis import load_gevis
from pecvd import gevis_training as training
GEVIS = load_gevis(ROOT)
display(GEVIS.describe())


In [ ]:
RUNTIMES, SAVED_HEADS = workflow['load_heads'](ROOT, GEVIS, DOMAIN, CHECKPOINTS)
display(SAVED_HEADS)
SESSIONS = {mode: training.Session(runtime) for mode, runtime in RUNTIMES.items()}


## 추론 연결 확인
참조 MIN/MAX의 중앙값을 만든 **인공 입력**으로 한 센서의 추론 연결만 확인합니다. 실제 Test 정확도나 정상 판정을 검증하는 실험은 아닙니다.


In [ ]:
for mode, runtime in RUNTIMES.items():
    REQUEST = workflow['example_request'](runtime)
    REPLY = SESSIONS[mode].process_window(**REQUEST)
    if REPLY['status'] != 'OK':
        raise RuntimeError(REPLY)
    print(mode, REPLY['status'])
    display(pd.DataFrame(REPLY['answers']))


실제 입력은 `SESSIONS[방식].process_window(...)`로 전달합니다. 입력 센서 순서·길이·공정 경로가 저장 메타데이터와 일치해야 합니다. 원본 단위 입력은 `standardized=False`, 이미 같은 전처리를 적용한 입력은 `True`를 사용합니다.
